### Reddit Hyperlink Network
#### Source: Stanford Network Analysis Project
##### Preliminary Data Exploration
##### McDevitt, Dixit, Jiang

In [20]:
!pip install torch_geometric
import os
import pandas as pd
import requests
import torch
import numpy as np
from torch_geometric.data import Data

from google.colab import drive


drive.mount('/content/drive')

path = "/content/drive/MyDrive/Thesis"


Mounted at /content/drive


#### Preliminary Data Setup

In [21]:
# The data location is at "snap.stanford.edu/data/soc-RedditHyperlinks.html"
file_path = "soc-redditHyperlinks-body.tsv"

df = pd.read_csv(file_path, sep='\t')


subreddits = pd.unique(df[['SOURCE_SUBREDDIT', 'TARGET_SUBREDDIT']].values.ravel())
sub_to_id = {name: i for i, name in enumerate(subreddits)}
num_nodes = len(subreddits)



edge_index = torch.tensor([
    [sub_to_id[src] for src in df['SOURCE_SUBREDDIT']],
    [sub_to_id[tgt] for tgt in df['TARGET_SUBREDDIT']]
], dtype=torch.long)


raw_features = np.array([np.fromstring(p, sep=',') for p in df['PROPERTIES']])
node_features = np.zeros((num_nodes, 86))
counts = np.zeros(num_nodes)

for i in range(len(df)):
    src_id = sub_to_id[df.iloc[i]['SOURCE_SUBREDDIT']]
    node_features[src_id] += raw_features[i]
    counts[src_id] += 1


counts[counts == 0] = 1
x = torch.tensor(node_features / counts[:, np.newaxis], dtype=torch.float)

print("5. Constructing PyTorch Geometric graph...")

graph_data = Data(x=x, edge_index=edge_index)

print("\n" + "="*40)
print("PREPROCESSING COMPLETE")
print(f"Nodes (Subreddits): {graph_data.num_nodes}")
print(f"Edges (Hyperlinks): {graph_data.num_edges}")
print(f"Feature Dimension:  {graph_data.num_node_features}")
print("="*40)

5. Constructing PyTorch Geometric graph...

PREPROCESSING COMPLETE
Nodes (Subreddits): 35776
Edges (Hyperlinks): 286561
Feature Dimension:  86


# Store and Health

In [19]:
import torch
import os



print(f"NaNs in features (X)? {torch.isnan(graph_data.x).any().item()}")


print(f"Has self-loops?       {graph_data.has_self_loops()}")
print(f"Has isolated nodes?   {graph_data.has_isolated_nodes()}")
print(f"Is graph directed?    {graph_data.is_directed()}")


out_degrees = torch.bincount(graph_data.edge_index[0], minlength=graph_data.num_nodes)
in_degrees = torch.bincount(graph_data.edge_index[1], minlength=graph_data.num_nodes)

print(f"Max Out-Degree:       {out_degrees.max().item()}")
print(f"Mean Out-Degree:      {out_degrees.float().mean().item():.2f}")
print(f"Nodes with 0 Out-Edges: {(out_degrees == 0).sum().item()}")


# Save the PyG Data object as is
file_name = 'reddit_base_graph.pt'
torch.save(graph_data, file_name)


print(f"File size: {os.path.getsize(file_name) / (1024 * 1024):.2f} MB")

NaNs in features (X)? False
Has self-loops?       False
Has isolated nodes?   False
Is graph directed?    True
Max Out-Degree:       4665
Mean Out-Degree:      8.01
Nodes with 0 Out-Edges: 7913
File size: 16.11 MB
